
Referential Integrity

In [0]:
%sql
select * from workforce.gold.fct_workforce f

Left join workforce.gold.dim_department d on f.department_key = d.department_key
where d.department_key is null

Business Mart Tables

1. attrition rate and avg tenure per department
2. attrition rate and avg tenure per job role
3. salary rankings

In [0]:
%sql
CREATE OR REPLACE TABLE workforce.gold.rpt_department_metrics AS

SELECT

    d.department,

    COUNT(*) AS headcount,

    ROUND(
        AVG(f.monthly_income),
        2
    ) AS avg_salary,

    ROUND(
        AVG(f.attrition) * 100,
        2
    ) AS attrition_rate_pct,

    ROUND(
        AVG(f.years_at_company),
        2
    ) AS avg_tenure

FROM workforce.gold.fct_workforce f

JOIN workforce.gold.dim_department d
ON f.department_key = d.department_key

GROUP BY d.department

In [0]:
%sql
CREATE OR REPLACE TABLE workforce.gold.rpt_job_role_metrics AS

SELECT

    j.job_role,

    COUNT(*) AS employees,

    ROUND(
        AVG(f.monthly_income),
        2
    ) AS avg_salary,

    ROUND(
        AVG(f.attrition) * 100,
        2
    ) AS attrition_rate_pct

FROM workforce.gold.fct_workforce f

JOIN workforce.gold.dim_job_role j
ON f.job_role_key = j.job_role_key

GROUP BY j.job_role

In [0]:
%sql
CREATE OR REPLACE TABLE workforce.gold.rpt_salary_rankings AS

SELECT

    employee_id,

    department_key,

    monthly_income,

    RANK() OVER (

        PARTITION BY department_key

        ORDER BY monthly_income DESC

    ) AS salary_rank

FROM workforce.gold.fct_workforce

Attrition Risk

In [0]:
%sql
CREATE OR REPLACE TABLE workforce.gold.fct_attrition_risk AS

SELECT

    employee_id,

    CASE

        WHEN over_time = 1
        AND job_satisfaction <= 2
        THEN 'High Risk'

        WHEN work_life_balance <= 2
        THEN 'Medium Risk'

        ELSE 'Low Risk'

    END AS attrition_risk

FROM workforce.gold.fct_workforce